In [ ]:
# =============================================================================
# Cell 0: NORTHSTAR + Config
# Tower 6: Tower of Hackers — Security QA for solace-browser
# =============================================================================

NORTHSTAR = "security-hackers-qa"
NOTEBOOK_PATH = "notebooks/qa/01-security-hackers-qa.ipynb"
TOWER = 6
TOWER_NAME = "Tower of Hackers"
TOTAL_FLOORS = 49
TOTAL_QUESTIONS = 343
TOTAL_PROBES = 22
MIN_SCORE = 70  # percent pass rate target
BASE_URL = "http://127.0.0.1:9222"
BROWSER_ROOT = "/home/phuc/projects/solace-browser"
SRC_ROOT = f"{BROWSER_ROOT}/src"
WEB_ROOT = f"{BROWSER_ROOT}/web"

# Results collector
RESULTS = []  # list of {"probe": str, "floor": int|str, "status": "PASS"|"FAIL", "detail": str}

def record(probe: str, floor, passed: bool, detail: str = ""):
    status = "PASS" if passed else "FAIL"
    RESULTS.append({"probe": probe, "floor": floor, "status": status, "detail": detail})
    print(f"{status}: [{probe}] (Floor {floor}) {detail}")

print(f"Tower {TOWER}: {TOWER_NAME}")
print(f"Total probes: {TOTAL_PROBES}")
print(f"Target pass rate: {MIN_SCORE}%")

In [ ]:
# =============================================================================
# Cell 1: Bootstrap — imports and path setup
# =============================================================================

import sys
import os
import json
import re
import hashlib
import subprocess
import importlib
import tempfile
import ast
from pathlib import Path
from urllib.request import urlopen, Request
from urllib.error import URLError, HTTPError

# Add solace-browser paths
if BROWSER_ROOT not in sys.path:
    sys.path.insert(0, BROWSER_ROOT)
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

def read_file(path: str) -> str:
    """Read file content as string."""
    return Path(path).read_text(encoding="utf-8")

def grep_files(pattern: str, root: str, glob_pat: str = "**/*.py") -> list:
    """Search for regex pattern in files matching glob."""
    hits = []
    for f in Path(root).glob(glob_pat):
        try:
            content = f.read_text(encoding="utf-8", errors="replace")
            for i, line in enumerate(content.splitlines(), 1):
                if re.search(pattern, line):
                    hits.append((str(f), i, line.strip()))
        except (OSError, UnicodeDecodeError):
            pass
    return hits

def http_request(url: str, method: str = "GET", data: dict = None, headers: dict = None) -> tuple:
    """Make HTTP request, return (status_code, headers_dict, body_str). Returns (-1, {}, error_str) on failure."""
    try:
        body = json.dumps(data).encode("utf-8") if data else None
        hdrs = headers or {}
        if data and "Content-Type" not in hdrs:
            hdrs["Content-Type"] = "application/json"
        req = Request(url, data=body, headers=hdrs, method=method)
        resp = urlopen(req, timeout=5)
        resp_headers = dict(resp.headers)
        resp_body = resp.read().decode("utf-8", errors="replace")
        return (resp.status, resp_headers, resp_body)
    except HTTPError as e:
        resp_headers = dict(e.headers) if e.headers else {}
        resp_body = e.read().decode("utf-8", errors="replace") if hasattr(e, "read") else str(e)
        return (e.code, resp_headers, resp_body)
    except (URLError, OSError) as e:
        return (-1, {}, str(e))

print("Bootstrap complete. Helpers: read_file(), grep_files(), http_request(), record()")

In [ ]:
# =============================================================================
# Probe 1: XSS — innerHTML usage audit (Floor 1: XSSHunter)
# Q: Is innerHTML used anywhere with server-supplied or user-supplied data?
# Q: Does the browser UI sanitize user input through template escaping?
# =============================================================================

# Scan all JS and HTML files for innerHTML assignments with dynamic content
innerHTML_hits = grep_files(r'innerHTML\s*[=+]', BROWSER_ROOT, '**/*.js')
innerHTML_hits += grep_files(r'innerHTML\s*[=+]', BROWSER_ROOT, '**/*.html')

# Filter: only flag lines that use variables (not static strings)
# Safe pattern: innerHTML = '<div>static</div>' (literal only)
# Unsafe pattern: innerHTML = variable, innerHTML += data, innerHTML = `${expr}`
unsafe_innerHTML = []
for filepath, lineno, line in innerHTML_hits:
    # Skip vendored/third-party files
    if '/vendor/' in filepath or '/node_modules/' in filepath:
        continue
    # Check for template literals with expressions or variable assignments
    if re.search(r'innerHTML\s*[=+].*\$\{', line):
        unsafe_innerHTML.append((filepath, lineno, line))
    elif re.search(r'innerHTML\s*[=+]\s*[a-zA-Z_]', line):
        # Variable assignment, not a string literal
        if not re.search(r'innerHTML\s*=\s*[\'"]', line):
            unsafe_innerHTML.append((filepath, lineno, line))

print(f"Total innerHTML usages found: {len(innerHTML_hits)}")
print(f"Potentially unsafe innerHTML (dynamic content): {len(unsafe_innerHTML)}")
for fp, ln, line in unsafe_innerHTML[:10]:
    rel = fp.replace(BROWSER_ROOT + '/', '')
    print(f"  {rel}:{ln}: {line[:120]}")

# Verdict: innerHTML with dynamic content is a risk vector
# PASS if zero unsafe usages, FAIL if any found
record("XSS-innerHTML", 1, len(unsafe_innerHTML) == 0,
       f"{len(unsafe_innerHTML)} unsafe innerHTML usages (dynamic content) in {len(innerHTML_hits)} total")

In [ ]:
# =============================================================================
# Probe 2: XSS — Content-Security-Policy headers (Floor 1 + 8 + 14)
# Q: Are Content-Security-Policy headers set with script-src restrictions?
# Q: Is CSP with script-src restriction present in browser pages?
# =============================================================================

# Check server code for CSP header setting
csp_hits = grep_files(r'Content-Security-Policy', BROWSER_ROOT, '**/*.py')
csp_html = grep_files(r'Content-Security-Policy', BROWSER_ROOT, '**/*.html')

print("CSP in Python files:")
for fp, ln, line in csp_hits:
    rel = fp.replace(BROWSER_ROOT + '/', '')
    print(f"  {rel}:{ln}: {line[:120]}")

print(f"\nCSP in HTML meta tags: {len(csp_html)}")
for fp, ln, line in csp_html:
    rel = fp.replace(BROWSER_ROOT + '/', '')
    print(f"  {rel}:{ln}: {line[:120]}")

# Check: CSP should be set on API responses with script-src
has_script_src = any('script-src' in line for _, _, line in csp_hits + csp_html)

# Also check security headers on main middleware
xfo_hits = grep_files(r'X-Frame-Options', BROWSER_ROOT, '**/*.py')
xcto_hits = grep_files(r'X-Content-Type-Options', BROWSER_ROOT, '**/*.py')

print(f"\nX-Frame-Options set: {len(xfo_hits) > 0}")
print(f"X-Content-Type-Options set: {len(xcto_hits) > 0}")

# CSP with script-src is critical for XSS defense
record("CSP-headers", "1,8,14", has_script_src,
       f"CSP script-src found: {has_script_src}. {len(csp_hits)} py hits, {len(csp_html)} html hits")

In [ ]:
# =============================================================================
# Probe 3: CORS misconfiguration (Floor 3: CSRFBreaker + Floor 16: CORSBuster)
# Q: Does the production CORS configuration restrict origins to a specific allowlist?
# Q: Are credentials (Access-Control-Allow-Credentials) combined with wildcard origins?
# =============================================================================

cors_hits = grep_files(r'Access-Control|CORS|cors', BROWSER_ROOT, '**/*.py')

print("CORS configuration in Python files:")
wildcard_origin = False
credentials_with_wildcard = False
for fp, ln, line in cors_hits:
    rel = fp.replace(BROWSER_ROOT + '/', '')
    print(f"  {rel}:{ln}: {line[:120]}")
    if "Allow-Origin" in line and "'*'" in line:
        wildcard_origin = True
    if "Allow-Credentials" in line and wildcard_origin:
        credentials_with_wildcard = True

# CORS wildcard * is dangerous for endpoints that handle auth/credentials
# Check if Access-Control-Allow-Credentials: true is combined with wildcard
creds_hits = grep_files(r'Allow-Credentials.*true', BROWSER_ROOT, '**/*.py')
if creds_hits and wildcard_origin:
    credentials_with_wildcard = True

print(f"\nWildcard origin (*): {wildcard_origin}")
print(f"Credentials with wildcard: {credentials_with_wildcard}")

# FAIL if wildcard origin is used (should be restrictive allowlist)
record("CORS-wildcard", "3,16", not wildcard_origin,
       f"Wildcard origin: {wildcard_origin}, Credentials+wildcard: {credentials_with_wildcard}")

In [ ]:
# =============================================================================
# Probe 4: Path Traversal (Floor 4: PathWalker)
# Q: Are all filesystem paths normalized and validated against a whitelist?
# Q: Can a crafted app_id with ../ sequences escape the intended directory?
# Q: Is Path.resolve().relative_to() used consistently?
# =============================================================================

# Test 1: Check inbox_outbox.py resolve_app_root for path traversal protection
io_code = read_file(f"{SRC_ROOT}/inbox_outbox.py")

# Look for the resolve + parent check pattern
has_resolve = '.resolve()' in io_code
has_parent_check = 'app_root.parent != self._apps_root' in io_code or 'relative_to' in io_code
print(f"inbox_outbox.py: resolve() used: {has_resolve}")
print(f"inbox_outbox.py: parent boundary check: {has_parent_check}")

# Test 2: Check budget_gates.py for path traversal
bg_code = read_file(f"{SRC_ROOT}/budget_gates.py")
bg_has_resolve = '.resolve()' in bg_code
print(f"budget_gates.py: resolve() used: {bg_has_resolve}")

# Test 3: Check fs_gateway for path traversal
fs_gw_path = Path(f"{SRC_ROOT}/fs_gateway/fs_gateway_service.py")
if fs_gw_path.exists():
    fs_code = fs_gw_path.read_text(encoding="utf-8")
    fs_has_resolve = 'resolve()' in fs_code and 'relative_to' in fs_code
    print(f"fs_gateway_service.py: resolve + relative_to: {fs_has_resolve}")
else:
    fs_has_resolve = False
    print("fs_gateway_service.py: NOT FOUND")

# Test 4: Grep for raw string concatenation in path construction (dangerous pattern)
raw_path_concat = grep_files(r'open\(.*\+.*\+', BROWSER_ROOT, '**/*.py')
raw_fstring_path = grep_files(r'open\(f[\'"]', BROWSER_ROOT, '**/*.py')
# Filter out test files
raw_path_concat = [(f, l, t) for f, l, t in raw_path_concat if '/test' not in f]
raw_fstring_path = [(f, l, t) for f, l, t in raw_fstring_path if '/test' not in f]

print(f"\nRaw path concatenation in open(): {len(raw_path_concat)}")
print(f"f-string path in open(): {len(raw_fstring_path)}")
for fp, ln, line in (raw_path_concat + raw_fstring_path)[:5]:
    rel = fp.replace(BROWSER_ROOT + '/', '')
    print(f"  {rel}:{ln}: {line[:120]}")

all_path_safe = has_resolve and has_parent_check and bg_has_resolve
record("PathTraversal-validation", 4, all_path_safe,
       f"resolve()={has_resolve}, parent_check={has_parent_check}, budget_resolve={bg_has_resolve}")

In [ ]:
# =============================================================================
# Probe 5: Clickjacking (Floor 5: ClickJacker)
# Q: Are X-Frame-Options: DENY headers set on all pages?
# Q: Are frame-busting scripts present as defense-in-depth?
# =============================================================================

# Check X-Frame-Options in server middleware
xfo_py = grep_files(r'X-Frame-Options', BROWSER_ROOT, '**/*.py')
xfo_html = grep_files(r'X-Frame-Options', BROWSER_ROOT, '**/*.html')

print("X-Frame-Options in Python:")
for fp, ln, line in xfo_py:
    rel = fp.replace(BROWSER_ROOT + '/', '')
    print(f"  {rel}:{ln}: {line[:120]}")

# Check for frame-busting scripts
frame_bust = grep_files(r'top\s*!==?\s*self|top\s*!==?\s*window|frameElement', BROWSER_ROOT, '**/*.js')
frame_bust_html = grep_files(r'top\s*!==?\s*self|top\s*!==?\s*window', BROWSER_ROOT, '**/*.html')

print(f"\nFrame-busting scripts in JS: {len(frame_bust)}")
print(f"Frame-busting in HTML: {len(frame_bust_html)}")

# Also check CSP frame-ancestors
frame_ancestors = grep_files(r'frame-ancestors', BROWSER_ROOT, '**/*.py')
print(f"CSP frame-ancestors: {len(frame_ancestors)}")

has_xfo = len(xfo_py) > 0
has_xfo_deny = any('DENY' in line for _, _, line in xfo_py)

record("Clickjacking-XFO", 5, has_xfo_deny,
       f"X-Frame-Options DENY: {has_xfo_deny}, frame-ancestors: {len(frame_ancestors)}, frame-bust: {len(frame_bust)}")

In [ ]:
# =============================================================================
# Probe 6: Deserialization safety (Floor 6: DeserialKill)
# Q: Is JSON the only serialization format used (no pickle, no YAML unsafe_load)?
# Q: Is json.loads used with standard library only?
# =============================================================================

# Check for dangerous deserialization
pickle_hits = grep_files(r'import pickle|pickle\.load|pickle\.loads', BROWSER_ROOT, '**/*.py')
yaml_unsafe = grep_files(r'yaml\.load\(|yaml\.unsafe_load', BROWSER_ROOT, '**/*.py')
eval_hits = grep_files(r'\beval\s*\(', BROWSER_ROOT, '**/*.py')
exec_hits = grep_files(r'\bexec\s*\(', BROWSER_ROOT, '**/*.py')

# Filter out test files and comments
def filter_non_test(hits):
    return [(f, l, t) for f, l, t in hits
            if '/test' not in f and not t.strip().startswith('#')]

pickle_prod = filter_non_test(pickle_hits)
yaml_unsafe_prod = filter_non_test(yaml_unsafe)
eval_prod = filter_non_test(eval_hits)
exec_prod = filter_non_test(exec_hits)

print(f"pickle usage (prod): {len(pickle_prod)}")
for fp, ln, line in pickle_prod:
    print(f"  {fp.replace(BROWSER_ROOT + '/', '')}:{ln}: {line[:100]}")

print(f"yaml.load / unsafe_load (prod): {len(yaml_unsafe_prod)}")
for fp, ln, line in yaml_unsafe_prod:
    print(f"  {fp.replace(BROWSER_ROOT + '/', '')}:{ln}: {line[:100]}")

print(f"eval() usage (prod): {len(eval_prod)}")
for fp, ln, line in eval_prod[:5]:
    print(f"  {fp.replace(BROWSER_ROOT + '/', '')}:{ln}: {line[:100]}")

print(f"exec() usage (prod): {len(exec_prod)}")
for fp, ln, line in exec_prod[:5]:
    print(f"  {fp.replace(BROWSER_ROOT + '/', '')}:{ln}: {line[:100]}")

# Also verify yaml.safe_load is used where yaml is present
yaml_safe = grep_files(r'yaml\.safe_load', BROWSER_ROOT, '**/*.py')
print(f"\nyaml.safe_load usage: {len(yaml_safe)}")

no_dangerous_deser = len(pickle_prod) == 0 and len(yaml_unsafe_prod) == 0
record("Deserialization-safety", 6, no_dangerous_deser,
       f"pickle={len(pickle_prod)}, yaml.load={len(yaml_unsafe_prod)}, eval={len(eval_prod)}, exec={len(exec_prod)}")

In [ ]:
from datetime import datetime, timezone, timedelta
# =============================================================================
# Probe 7: OAuth3 scope enforcement (Floor 7 + 9: PrivEscalator + ScopeCreep)
# Q: Are OAuth3 scopes enforced at every endpoint?
# Q: Would a crafted scope string bypass the scope parser?
# Q: Is role-based access control enforced at every API boundary?
# =============================================================================

# Import OAuth3 modules
try:
    from oauth3.enforcement import ScopeGate, GateResult, GATE_PASS, GATE_BLOCKED
    from oauth3.token import AgencyToken
    from oauth3.revocation import TokenStore
    from oauth3.scopes import validate_scopes, SCOPES
    oauth3_available = True
    print("OAuth3 modules loaded successfully")
except ImportError as e:
    oauth3_available = False
    print(f"OAuth3 import failed: {e}")

if oauth3_available:
    # Test 1: Create token with valid scopes
    token = AgencyToken(
        token_id="test-token-008",
        issuer="https://test.solaceagi.com",
        subject="user:test@example.com",
        scopes=("gmail.read.inbox",),
        intent="test scope enforcement",
        issued_at=datetime.now(timezone.utc).isoformat(),
        expires_at=(datetime.now(timezone.utc) + timedelta(hours=1)).isoformat(),
    )
    store = TokenStore()
    store.add(token)

    # Test 2: Scope gate should PASS for matching scope
    gate = ScopeGate(token, required_scopes=["gmail.read.inbox"], store=store)
    result = gate.check_all()
    scope_pass = result.allowed
    print(f"Matching scope check: allowed={result.allowed}")

    # Test 3: Scope gate should BLOCK for non-matching scope
    gate2 = ScopeGate(token, required_scopes=["linkedin.post.text"], store=store)
    result2 = gate2.check_all()
    scope_block = not result2.allowed
    print(f"Non-matching scope check: blocked={not result2.allowed}")

    # Test 4: None token should be blocked
    gate3 = ScopeGate(None, required_scopes=["gmail.read.inbox"], store=store)
    result3 = gate3.check_all()
    null_block = not result3.allowed
    print(f"None token check: blocked={not result3.allowed}")

    # Test 5: Crafted scope string (injection attempt)
    try:
        bad_scopes = validate_scopes(["gmail.read.inbox; DROP TABLE"])
        scope_injection_blocked = False  # Should have raised
        print(f"Scope injection test: NOT blocked (bad_scopes={bad_scopes})")
    except (ValueError, TypeError) as e:
        scope_injection_blocked = True
        print(f"Scope injection test: blocked ({e})")

    all_scope_pass = scope_pass and scope_block and null_block
    record("OAuth3-scope-enforcement", "7,9", all_scope_pass,
           f"match={scope_pass}, block={scope_block}, null_block={null_block}, injection_blocked={scope_injection_blocked}")
else:
    record("OAuth3-scope-enforcement", "7,9", False, "OAuth3 modules not importable")

In [ ]:
# =============================================================================
# Probe 8: Security Headers (Floor 8: HeaderPoison)
# Q: Is X-Content-Type-Options: nosniff set?
# Q: Are response headers set exclusively by the framework?
# Q: Is Cache-Control set appropriately?
# =============================================================================

# Check security headers in server middleware
server_code = read_file(f"{BROWSER_ROOT}/solace_browser_server.py")

headers_found = {}
for header in ['X-Content-Type-Options', 'X-Frame-Options', 'Strict-Transport-Security',
               'Cache-Control', 'Content-Security-Policy', 'X-XSS-Protection']:
    hits = grep_files(re.escape(header), BROWSER_ROOT, '**/*.py')
    headers_found[header] = len(hits) > 0
    status = 'SET' if headers_found[header] else 'MISSING'
    print(f"  {header}: {status} ({len(hits)} locations)")

# Critical: nosniff must be present
has_nosniff = any('nosniff' in line for _, _, line in
                  grep_files(r'nosniff', BROWSER_ROOT, '**/*.py'))

# Check for user-controllable header injection
header_inject = grep_files(r'response\.headers\[.*request', BROWSER_ROOT, '**/*.py')
print(f"\nUser-controllable headers: {len(header_inject)}")
for fp, ln, line in header_inject:
    print(f"  {fp.replace(BROWSER_ROOT + '/', '')}:{ln}: {line[:100]}")

critical_headers = headers_found.get('X-Content-Type-Options', False) and \
                   headers_found.get('X-Frame-Options', False)
record("Security-headers", 8, critical_headers and has_nosniff,
       f"nosniff={has_nosniff}, XFO={headers_found.get('X-Frame-Options')}, " +
       f"HSTS={headers_found.get('Strict-Transport-Security')}, user-inject={len(header_inject)}")

In [ ]:
# =============================================================================
# Probe 9: Token Security — Forgery + Expiry (Floor 11 + 18: TokenForger + ReplayAttacker)
# Q: Are tokens signed with SHA-256 and is alg:none prevented?
# Q: Would a token with a modified payload pass signature verification?
# Q: Can an intercepted request be replayed within token TTL?
# =============================================================================

if oauth3_available:
    from oauth3.token import AgencyToken, DEFAULT_TTL_SECONDS, MAX_TTL_SECONDS
    from datetime import timedelta, datetime, timezone

    # Test 1: Token has signature_stub (SHA-256 hash)
    token = AgencyToken(
        token_id="test-token-010",
        issuer="https://test.solaceagi.com",
        subject="user:test@example.com",
        scopes=("gmail.read.inbox",),
        intent="test token integrity",
        issued_at=datetime.now(timezone.utc).isoformat(),
        expires_at=(datetime.now(timezone.utc) + timedelta(hours=1)).isoformat(),
    )
    has_sig = hasattr(token, 'signature_stub') and token.signature_stub
    is_sha256 = has_sig and token.signature_stub.startswith('sha256:')
    print(f"Token has signature_stub: {has_sig}")
    print(f"Signature is SHA-256: {is_sha256}")
    print(f"Signature: {token.signature_stub[:60]}..." if has_sig else "No signature")

    # Test 2: Verify modified payload changes signature
    token2 = AgencyToken(
        token_id="test-token-010b",
        issuer="https://test.solaceagi.com",
        subject="user:test@example.com",
        scopes=["gmail.read.inbox", "linkedin.post.text"],  # extra scope
        intent="test token integrity",
        issued_at=datetime.now(timezone.utc).isoformat(),
        expires_at=(datetime.now(timezone.utc) + timedelta(hours=1)).isoformat(),
    )
    sigs_differ = (not has_sig) or (token.signature_stub != token2.signature_stub)
    print(f"Modified payload produces different signature: {sigs_differ}")

    # Test 3: Token expiry enforcement
    expired_token = AgencyToken(
        token_id="test-token-010c",
        issuer="https://test.solaceagi.com",
        subject="user:test@example.com",
        scopes=["gmail.read.inbox"],
        intent="expired test",
        issued_at=datetime.now(timezone.utc).isoformat(),
        expires_at=(datetime.now(timezone.utc) - timedelta(hours=1)).isoformat(),  # already expired
    )
    store = TokenStore()
    store.add(expired_token)
    gate = ScopeGate(expired_token, required_scopes=["gmail.read.inbox"], store=store)
    result = gate.check_all()
    expired_blocked = not result.allowed
    print(f"Expired token blocked: {expired_blocked}")

    # Test 4: TTL limits
    print(f"Default TTL: {DEFAULT_TTL_SECONDS}s ({DEFAULT_TTL_SECONDS/3600}h)")
    print(f"Max TTL: {MAX_TTL_SECONDS}s ({MAX_TTL_SECONDS/3600}h)")
    ttl_reasonable = MAX_TTL_SECONDS <= 86400  # 24 hours max

    all_pass = is_sha256 and sigs_differ and expired_blocked and ttl_reasonable
    record("Token-integrity", "11,18", all_pass,
           f"sha256={is_sha256}, sigs_differ={sigs_differ}, expired_blocked={expired_blocked}, ttl_max={MAX_TTL_SECONDS}s")
else:
    record("Token-integrity", "11,18", False, "OAuth3 modules not importable")

In [ ]:
# =============================================================================
# Probe 10: Vault Encryption (Floor 27: CryptoBreaker)
# Q: Is AES-256-GCM used for vault encryption with proper IV handling?
# Q: Is the vault key derivation seed readable by any process?
# Q: Is there a lock file to prevent concurrent access corruption?
# =============================================================================

vault_path = f"{SRC_ROOT}/oauth3/vault.py"
vault_code = read_file(vault_path)

# Check encryption algorithm
has_aesgcm = 'AESGCM' in vault_code
has_aes256 = '32' in vault_code  # 32 bytes = 256 bits
print(f"AES-GCM present: {has_aesgcm}")

# Check IV/nonce generation
has_random_nonce = 'os.urandom' in vault_code or 'secrets.token_bytes' in vault_code
print(f"Random nonce generation: {has_random_nonce}")

# Check key file permissions
has_chmod_600 = '0o600' in vault_code or '0600' in vault_code
print(f"Key file chmod 600: {has_chmod_600}")

# Check for concurrent access protection
has_lock = 'fcntl' in vault_code or 'lock' in vault_code.lower() or 'Lock' in vault_code
print(f"Concurrent access protection: {has_lock}")

# Check key generation
key_gen_secure = 'secrets.token_bytes' in vault_code
print(f"Secure key generation (secrets.token_bytes): {key_gen_secure}")

vault_secure = has_aesgcm and has_chmod_600 and key_gen_secure
record("Vault-encryption", 27, vault_secure,
       f"AES-GCM={has_aesgcm}, chmod600={has_chmod_600}, secure_keygen={key_gen_secure}, lock={has_lock}")

In [ ]:
# =============================================================================
# Probe 11: Auth Proxy — Fail-Closed (Floor 7 + 10 + 12: PrivEscalator + SessionThief + CredStuffer)
# Q: Do all API endpoints require Authorization headers?
# Q: Is the auth proxy fail-closed (missing token -> 401)?
# Q: Are tokens validated in constant time?
# =============================================================================

auth_proxy_code = read_file(f"{SRC_ROOT}/auth_proxy.py")

# Check fail-closed design
has_fail_closed = 'FAIL-CLOSED' in auth_proxy_code or 'fail-closed' in auth_proxy_code.lower()
has_401 = '401' in auth_proxy_code
has_unauthorized = 'unauthorized' in auth_proxy_code.lower()

print(f"Fail-closed documented: {has_fail_closed}")
print(f"401 response code: {has_401}")
print(f"Unauthorized error: {has_unauthorized}")

# Check token format validation
has_token_format = '_TOKEN_FORMAT_RE' in auth_proxy_code or 'TOKEN_PREFIX' in auth_proxy_code
print(f"Token format validation: {has_token_format}")

# Check for timing-safe comparison
hmac_hits = grep_files(r'hmac\.compare_digest|secrets\.compare_digest', BROWSER_ROOT, '**/*.py')
has_timing_safe = len(hmac_hits) > 0
print(f"Timing-safe comparison: {has_timing_safe} ({len(hmac_hits)} locations)")
for fp, ln, line in hmac_hits:
    print(f"  {fp.replace(BROWSER_ROOT + '/', '')}:{ln}: {line[:100]}")

# Check session token TTL
session_ttl_match = re.search(r'SESSION_TOKEN_TTL_SECONDS\s*=\s*(\d+)', auth_proxy_code)
if session_ttl_match:
    session_ttl = int(session_ttl_match.group(1))
    print(f"Session token TTL: {session_ttl}s ({session_ttl/60}min)")
    ttl_ok = session_ttl <= 600  # 10 minutes max is reasonable
else:
    ttl_ok = False
    print("Session token TTL: NOT FOUND")

auth_proxy_ok = has_fail_closed and has_401 and has_token_format
record("AuthProxy-fail-closed", "7,10,12", auth_proxy_ok,
       f"fail-closed={has_fail_closed}, 401={has_401}, token_format={has_token_format}, timing_safe={has_timing_safe}")

In [ ]:
# =============================================================================
# Probe 12: Evidence Chain Integrity (Floor 32: LogTamper)
# Q: Can the evidence chain be replaced with a forged chain that passes verification?
# Q: Does the hash chain detect tampering but fail to prevent deletion?
# Q: Would recomputing all hashes create an undetectable forged chain?
# =============================================================================

try:
    from oauth3.evidence import EvidenceChain, GENESIS_HASH
    evidence_available = True
    print("EvidenceChain loaded")
except ImportError as e:
    evidence_available = False
    print(f"EvidenceChain import failed: {e}")

if evidence_available:
    with tempfile.TemporaryDirectory() as tmpdir:
        chain = EvidenceChain(Path(tmpdir) / "test_chain.jsonl")

        # Test 1: Create a chain and verify
        h1 = chain.log_event("token_issued", {"token_id": "t1", "scopes": ["gmail.read"]})
        h2 = chain.log_event("token_used", {"token_id": "t1", "action": "read_inbox"})
        h3 = chain.log_event("token_revoked", {"token_id": "t1", "reason": "manual"})

        valid, error = chain.verify_chain()
        print(f"Chain valid after 3 events: {valid} (error: {error})")

        # Test 2: Tamper with the chain (modify middle event)
        chain_file = Path(tmpdir) / "test_chain.jsonl"
        lines = chain_file.read_text().strip().split('\n')
        tampered = json.loads(lines[1])
        tampered['data']['action'] = 'DELETE_ALL_DATA'  # malicious modification
        lines[1] = json.dumps(tampered, sort_keys=True)
        chain_file.write_text('\n'.join(lines) + '\n')

        chain2 = EvidenceChain(chain_file)
        valid2, error2 = chain2.verify_chain()
        tamper_detected = not valid2
        print(f"Tamper detected: {tamper_detected} (error: {error2})")

        # Test 3: Check if full chain replacement is detectable
        # A forged chain with recomputed hashes SHOULD still pass verification
        # This is an expected weakness without external anchoring
        forged_chain = EvidenceChain(Path(tmpdir) / "forged_chain.jsonl")
        forged_chain.log_event("token_issued", {"token_id": "t1", "scopes": ["admin.all"]})
        valid3, _ = forged_chain.verify_chain()
        print(f"Forged chain passes verification: {valid3} (expected weakness without external anchor)")

        # Test 4: Genesis hash is well-defined
        genesis_ok = GENESIS_HASH == "0" * 64
        print(f"Genesis hash: {GENESIS_HASH[:20]}... (64 zeros: {genesis_ok})")

    record("EvidenceChain-integrity", 32, valid and tamper_detected,
           f"valid={valid}, tamper_detected={tamper_detected}, forged_passes={valid3} (no external anchor)")
else:
    record("EvidenceChain-integrity", 32, False, "EvidenceChain not importable")

In [ ]:
from datetime import datetime, timezone, timedelta
# =============================================================================
# Probe 13: Token Revocation Immediacy (Floor 9: ScopeCreep)
# Q: Is revocation immediate across all three channels?
# Q: Does scope enforcement survive concurrent request during scope modification?
# =============================================================================

if oauth3_available:
    from oauth3.token import AgencyToken
    from oauth3.revocation import TokenStore
    from oauth3.enforcement import ScopeGate

    store = TokenStore()
    token = AgencyToken(
        token_id="test-token-014",
        issuer="https://test.solaceagi.com",
        subject="user:test@example.com",
        scopes=("gmail.read.inbox",),
        intent="revocation test",
        issued_at=datetime.now(timezone.utc).isoformat(),
        expires_at=(datetime.now(timezone.utc) + timedelta(hours=1)).isoformat(),
    )
    store.add(token)

    # Test 1: Token works before revocation
    gate = ScopeGate(token, required_scopes=["gmail.read.inbox"], store=store)
    pre_revoke = gate.check_all()
    print(f"Before revocation: allowed={pre_revoke.allowed}")

    # Test 2: Revoke and verify immediate effect
    store.revoke(token.token_id)
    revoked_token = store.get(token.token_id)
    is_revoked = revoked_token is not None and revoked_token.revoked
    print(f"Token revoked flag: {is_revoked}")

    # Test 3: Scope gate blocks revoked token
    gate2 = ScopeGate(revoked_token, required_scopes=["gmail.read.inbox"], store=store)
    post_revoke = gate2.check_all()
    revoke_blocks = not post_revoke.allowed
    print(f"After revocation: blocked={revoke_blocks}")
    if not post_revoke.allowed and post_revoke.blocking_gate:
        print(f"  Blocking gate: {post_revoke.blocking_gate.gate} - {post_revoke.blocking_gate.error_detail}")

    record("Token-revocation", 9, pre_revoke.allowed and revoke_blocks,
           f"pre_allowed={pre_revoke.allowed}, post_blocked={revoke_blocks}, revoked_flag={is_revoked}")
else:
    record("Token-revocation", 9, False, "OAuth3 modules not importable")

In [ ]:
# =============================================================================
# Probe 14: API Evaluate endpoint — JS injection (Floor 1 + 19: XSSHunter + PromptInjector)
# Q: Does /api/evaluate sanitize the expression parameter?
# Q: Can a crafted expression exfiltrate data?
# =============================================================================

# Static analysis: Check if /api/evaluate has any input validation
server_code = read_file(f"{BROWSER_ROOT}/solace_browser_server.py")

# Find the evaluate handler
eval_handler_match = re.search(
    r'async def _handle_evaluate.*?(?=async def |$)',
    server_code,
    re.DOTALL
)

if eval_handler_match:
    eval_handler = eval_handler_match.group(0)
    print("_handle_evaluate handler found:")
    for line in eval_handler.strip().split('\n')[:10]:
        print(f"  {line}")

    # Check for input validation/sanitization
    has_sanitize = bool(re.search(r'sanitiz|validat|allowlist|whitelist|escape|filter', eval_handler, re.I))
    has_auth_check = bool(re.search(r'oauth3|bearer|token|auth', eval_handler, re.I))
    raw_eval = 'evaluate(expression)' in eval_handler or 'evaluate(script)' in eval_handler

    print(f"\nInput sanitization: {has_sanitize}")
    print(f"Auth check: {has_auth_check}")
    print(f"Raw expression passed to evaluate: {raw_eval}")

    # /api/evaluate passes raw user input to browser.evaluate() without sanitization
    # This is inherently dangerous — any JS can be executed in the browser context
    record("API-evaluate-sanitization", "1,19", has_sanitize or has_auth_check,
           f"sanitize={has_sanitize}, auth={has_auth_check}, raw_eval={raw_eval}")
else:
    print("_handle_evaluate handler NOT FOUND")
    record("API-evaluate-sanitization", "1,19", False, "Handler not found")

In [ ]:
# =============================================================================
# Probe 15: Budget Gates — Fail-Closed (Floor 7 + 24: PrivEscalator + AgentEscaper)
# Q: Are budget gates fail-closed (missing policy -> BLOCKED)?
# Q: Does the approval gate cover ALL tool calls?
# =============================================================================

try:
    from budget_gates import (
        BudgetGateChecker,
        BudgetPolicyNotFoundError,
        BudgetExhaustedError,
        DomainNotAllowedError,
    )
    budget_available = True
    print("BudgetGateChecker loaded")
except ImportError as e:
    budget_available = False
    print(f"BudgetGateChecker import failed: {e}")

if budget_available:
    with tempfile.TemporaryDirectory() as tmpdir:
        apps_root = Path(tmpdir) / "apps"
        apps_root.mkdir()

        checker = BudgetGateChecker(apps_root)

        # Test 1: Non-existent app -> BLOCKED
        result = checker.check_all("nonexistent-app", "example.com")
        missing_blocked = not result["allowed"]
        print(f"Missing app blocked: {missing_blocked} ({result.get('reason', '')[:80]})")

        # Test 2: App without policy file -> BLOCKED
        app_dir = apps_root / "test-app"
        app_dir.mkdir()
        (app_dir / "manifest.yaml").write_text("name: test\ntype: worker\n")
        result2 = checker.check_all("test-app", "example.com")
        no_policy_blocked = not result2["allowed"]
        print(f"No policy file blocked: {no_policy_blocked} ({result2.get('reason', '')[:80]})")

    record("BudgetGates-fail-closed", "7,24", missing_blocked and no_policy_blocked,
           f"missing_app={missing_blocked}, no_policy={no_policy_blocked}")
else:
    record("BudgetGates-fail-closed", "7,24", False, "BudgetGateChecker not importable")

In [ ]:
# =============================================================================
# Probe 16: Capture Pipeline — Domain exclusion (Floor 23 + 33: EmbedExfiltrator + CachePoisoner)
# Q: Does the capture pipeline exclude sensitive domains (localhost, chrome://)?
# Q: Does RTC (round-trip check) prevent corrupted captures?
# =============================================================================

try:
    from capture_pipeline import (
        DomainExcludedError,
        RTCVerificationError,
        CaptureNotFoundError,
    )
    capture_available = True
    print("capture_pipeline loaded")
except ImportError as e:
    capture_available = False
    print(f"capture_pipeline import failed: {e}")

# Static analysis of capture pipeline code
cap_code = read_file(f"{SRC_ROOT}/capture_pipeline.py")

# Check domain exclusion
has_domain_exclude = 'DomainExcludedError' in cap_code
has_localhost_check = 'localhost' in cap_code or '127.0.0.1' in cap_code
has_chrome_check = 'chrome://' in cap_code or 'chrome-extension' in cap_code
has_ip_check = 'ipaddress' in cap_code  # private IP checking

print(f"Domain exclusion error: {has_domain_exclude}")
print(f"Localhost check: {has_localhost_check}")
print(f"Chrome protocol check: {has_chrome_check}")
print(f"IP address validation: {has_ip_check}")

# Check RTC (round-trip check)
has_rtc = 'RTCVerificationError' in cap_code
has_sha256_verify = 'sha256' in cap_code.lower()
print(f"RTC verification: {has_rtc}")
print(f"SHA-256 hash verify: {has_sha256_verify}")

# Check Fallback Ban compliance
broad_except = re.findall(r'except\s+Exception\s*:', cap_code)
bare_except = re.findall(r'except\s*:', cap_code)
print(f"Broad except clauses: {len(broad_except)}")
print(f"Bare except clauses: {len(bare_except)}")

capture_safe = has_domain_exclude and has_rtc and len(broad_except) == 0 and len(bare_except) == 0
record("CapturePipeline-safety", "23,33", capture_safe,
       f"domain_exclude={has_domain_exclude}, rtc={has_rtc}, ip_check={has_ip_check}, " +
       f"broad_except={len(broad_except)}, bare_except={len(bare_except)}")

In [ ]:
# =============================================================================
# Probe 17: Fallback Ban compliance (Floor 48+49: Security Architect + Zero-Day)
# Q: Are there broad except clauses that silently swallow errors?
# Q: Is the codebase Fallback Ban compliant?
# =============================================================================

# Scan ALL Python files for Fallback Ban violations
broad_except_all = grep_files(r'except\s+Exception\s*:', BROWSER_ROOT, '**/*.py')
bare_except_all = grep_files(r'^\s*except\s*:', BROWSER_ROOT, '**/*.py')
except_pass = grep_files(r'except.*:\s*$', BROWSER_ROOT, '**/*.py')  # except followed by pass on next line

# Filter out test files
broad_prod = [(f, l, t) for f, l, t in broad_except_all
              if '/test' not in f and '__pycache__' not in f]
bare_prod = [(f, l, t) for f, l, t in bare_except_all
             if '/test' not in f and '__pycache__' not in f]

print(f"Broad 'except Exception:' in prod code: {len(broad_prod)}")
for fp, ln, line in broad_prod[:15]:
    rel = fp.replace(BROWSER_ROOT + '/', '')
    print(f"  {rel}:{ln}: {line[:100]}")

print(f"\nBare 'except:' in prod code: {len(bare_prod)}")
for fp, ln, line in bare_prod[:10]:
    rel = fp.replace(BROWSER_ROOT + '/', '')
    print(f"  {rel}:{ln}: {line[:100]}")

# Also check for return None/empty in except blocks
silent_swallow = grep_files(r'except.*Exception.*:\s*$', BROWSER_ROOT, '**/*.py')

# Verdict: Fallback Ban requires zero broad except in critical modules
# Allow some in non-critical UI code, but flag src/ violations
src_broad = [(f, l, t) for f, l, t in broad_prod if '/src/' in f]
print(f"\nBroad except in src/: {len(src_broad)}")
for fp, ln, line in src_broad[:10]:
    rel = fp.replace(BROWSER_ROOT + '/', '')
    print(f"  {rel}:{ln}: {line[:100]}")

record("FallbackBan-compliance", "48,49", len(src_broad) == 0,
       f"src/ broad except: {len(src_broad)}, total prod broad: {len(broad_prod)}, bare: {len(bare_prod)}")

In [ ]:
# =============================================================================
# Probe 18: Supply Chain — Dependency pinning (Floor 30: SupplyChain)
# Q: Do dependency lock files with hashes exist?
# Q: Are all dependency versions pinned?
# =============================================================================

# Check for lock files
lock_files = [
    ("requirements.txt", Path(f"{BROWSER_ROOT}/requirements.txt")),
    ("requirements-dev.txt", Path(f"{BROWSER_ROOT}/requirements-dev.txt")),
    ("poetry.lock", Path(f"{BROWSER_ROOT}/poetry.lock")),
    ("Pipfile.lock", Path(f"{BROWSER_ROOT}/Pipfile.lock")),
    ("pyproject.toml", Path(f"{BROWSER_ROOT}/pyproject.toml")),
]

found_locks = []
for name, path in lock_files:
    exists = path.exists()
    print(f"{name}: {'EXISTS' if exists else 'MISSING'}")
    if exists:
        found_locks.append(name)

# Check if requirements.txt has pinned versions (== not >=)
has_pinned = False
has_hashes = False
unpinned_deps = []
req_path = Path(f"{BROWSER_ROOT}/requirements.txt")
if req_path.exists():
    req_content = req_path.read_text()
    for line in req_content.strip().split('\n'):
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('-'):
            continue
        if '==' in line:
            has_pinned = True
        elif '>=' in line or line.isalpha() or (line and '==' not in line and '>=' not in line):
            unpinned_deps.append(line)
        if '--hash' in line:
            has_hashes = True

    print(f"\nPinned versions (==): {has_pinned}")
    print(f"Hash verification (--hash): {has_hashes}")
    print(f"Unpinned dependencies: {len(unpinned_deps)}")
    for dep in unpinned_deps[:10]:
        print(f"  {dep}")

record("SupplyChain-pinning", 30, has_pinned and len(found_locks) > 0,
       f"lock_files={found_locks}, pinned={has_pinned}, hashes={has_hashes}, unpinned={len(unpinned_deps)}")

In [ ]:
# =============================================================================
# Probe 19: Prompt Injection Defense (Floor 19: PromptInjector)
# Q: Is there ANY input sanitization on user prompts before LLM processing?
# Q: Are tool-use calls validated against a whitelist?
# Q: Does dual-LLM verification exist?
# =============================================================================

# Check for prompt injection defenses
prompt_sanitize = grep_files(r'sanitiz.*prompt|prompt.*sanitiz|injection.*detect|detect.*injection',
                              BROWSER_ROOT, '**/*.py')
prompt_filter = grep_files(r'prompt.*filter|filter.*prompt|input.*validation.*llm',
                            BROWSER_ROOT, '**/*.py')
tool_whitelist = grep_files(r'tool.*whitelist|whitelist.*tool|allowed_tools|tool.*allowlist',
                             BROWSER_ROOT, '**/*.py')
dual_llm = grep_files(r'dual.*llm|llm.*verify|second.*model|cross.*validate',
                       BROWSER_ROOT, '**/*.py')

print(f"Prompt sanitization: {len(prompt_sanitize)} hits")
for fp, ln, line in prompt_sanitize[:5]:
    print(f"  {fp.replace(BROWSER_ROOT + '/', '')}:{ln}: {line[:100]}")

print(f"Prompt filtering: {len(prompt_filter)} hits")
print(f"Tool whitelist: {len(tool_whitelist)} hits")
print(f"Dual-LLM verification: {len(dual_llm)} hits")

# Check plugin sandbox
sandbox_path = Path(f"{SRC_ROOT}/plugins/sandbox.py")
if sandbox_path.exists():
    sandbox_code = sandbox_path.read_text(encoding="utf-8")
    has_no_exec = 'no exec' in sandbox_code.lower() or 'exec' in sandbox_code
    has_no_eval = 'no eval' in sandbox_code.lower() or 'eval' in sandbox_code
    has_no_pickle = 'no pickle' in sandbox_code.lower() or 'pickle' in sandbox_code
    print(f"\nPlugin sandbox: exec banned={has_no_exec}, eval banned={has_no_eval}, pickle banned={has_no_pickle}")
else:
    print("\nPlugin sandbox: NOT FOUND")

has_defense = len(prompt_sanitize) > 0 or len(tool_whitelist) > 0
record("PromptInjection-defense", 19, has_defense,
       f"sanitize={len(prompt_sanitize)}, filter={len(prompt_filter)}, " +
       f"tool_whitelist={len(tool_whitelist)}, dual_llm={len(dual_llm)}")

In [ ]:
# =============================================================================
# Probe 20: Sensitive Data in Error Responses (Floor 34 + 48: SideChannel + Security Architect)
# Q: Do error messages reflect unsanitized input or leak internal paths?
# Q: Is there protection against timing oracle attacks?
# =============================================================================

# Check for stack trace exposure in error handlers
traceback_hits = grep_files(r'traceback\.format_exc|traceback\.print_exc|import traceback',
                             BROWSER_ROOT, '**/*.py')
traceback_prod = [(f, l, t) for f, l, t in traceback_hits if '/test' not in f]

print(f"Traceback usage in prod code: {len(traceback_prod)}")
for fp, ln, line in traceback_prod[:10]:
    rel = fp.replace(BROWSER_ROOT + '/', '')
    print(f"  {rel}:{ln}: {line[:100]}")

# Check for str(e) in response bodies (could leak internal details)
error_leak = grep_files(r'str\(e\)|str\(exc\)|str\(error\)', BROWSER_ROOT, 'solace_browser_server.py')
print(f"\nstr(exception) in server responses: {len(error_leak)}")
for fp, ln, line in error_leak[:10]:
    print(f"  line {ln}: {line[:100]}")

# Check for DEBUG mode control
debug_hits = grep_files(r'DEBUG|debug.*=.*True|debug.*mode', BROWSER_ROOT, 'solace_browser_server.py')
print(f"\nDEBUG references in server: {len(debug_hits)}")

# Error responses should use generic messages, not raw exception strings
record("ErrorInfoLeak", "34,48", len(error_leak) < 5,
       f"str(exception) in responses: {len(error_leak)}, traceback in prod: {len(traceback_prod)}")

In [ ]:
# =============================================================================
# Probe 21: Vulnerability Disclosure (Floor 40 + 42: ZeroDayBroker + BugBountyHunter)
# Q: Is security.txt deployed per RFC 9116?
# Q: Is there a responsible disclosure policy?
# Q: Does a bug bounty program exist?
# =============================================================================

# Check for security.txt
security_txt_paths = [
    Path(f"{BROWSER_ROOT}/web/.well-known/security.txt"),
    Path(f"{BROWSER_ROOT}/static/.well-known/security.txt"),
    Path(f"{BROWSER_ROOT}/.well-known/security.txt"),
    Path(f"{BROWSER_ROOT}/web/security.txt"),
]

security_txt_found = False
for p in security_txt_paths:
    if p.exists():
        print(f"security.txt found: {p}")
        print(p.read_text()[:500])
        security_txt_found = True
        break

if not security_txt_found:
    print("security.txt: NOT FOUND at any standard location")

# Check for disclosure policy
disclosure_files = list(Path(BROWSER_ROOT).glob('**/SECURITY*'))
disclosure_files += list(Path(BROWSER_ROOT).glob('**/security*policy*'))
print(f"\nSecurity/disclosure files: {len(disclosure_files)}")
for f in disclosure_files:
    print(f"  {f}")

# Check for security contact in README or docs
security_contact = grep_files(r'security.*@|vulnerability.*report|bug.*bounty|responsible.*disclosure',
                               BROWSER_ROOT, '**/*.md')
print(f"\nSecurity references in docs: {len(security_contact)}")

record("Disclosure-policy", "40,42", security_txt_found or len(disclosure_files) > 0,
       f"security.txt={security_txt_found}, security_files={len(disclosure_files)}, doc_refs={len(security_contact)}")

In [ ]:
# =============================================================================
# Probe 22: Live Server Probes (Floors 1-8, 12, 16, 34)
# Requires solace-browser running on port 9222
# Tests actual HTTP behavior: headers, auth, error handling
# =============================================================================

print("Testing live server at", BASE_URL)
print("(If server is not running, probes will report connection failures)\n")

# Test 1: Health endpoint accessibility
status, headers, body = http_request(f"{BASE_URL}/api/health")
server_up = status == 200
print(f"Health check: status={status}, up={server_up}")

if server_up:
    # Test 2: Security headers on responses
    xfo = headers.get('X-Frame-Options', 'MISSING')
    xcto = headers.get('X-Content-Type-Options', 'MISSING')
    acao = headers.get('Access-Control-Allow-Origin', 'MISSING')
    print(f"X-Frame-Options: {xfo}")
    print(f"X-Content-Type-Options: {xcto}")
    print(f"Access-Control-Allow-Origin: {acao}")

    # Test 3: XSS payload in /api/fill (should not reflect)
    xss_payload = {"selector": "<script>alert(1)</script>", "text": "test"}
    xss_status, xss_headers, xss_body = http_request(
        f"{BASE_URL}/api/fill", method="POST", data=xss_payload
    )
    xss_reflected = '<script>alert(1)</script>' in xss_body
    print(f"\nXSS reflection test: status={xss_status}, reflected={xss_reflected}")

    # Test 4: Path traversal in URL
    traversal_status, _, traversal_body = http_request(
        f"{BASE_URL}/api/artifacts/../../etc/passwd"
    )
    traversal_success = 'root:' in traversal_body
    print(f"Path traversal test: status={traversal_status}, leaked={traversal_success}")

    # Test 5: Error info leakage
    err_status, _, err_body = http_request(
        f"{BASE_URL}/api/evaluate", method="POST",
        data={"expression": "throw new Error('test')"}
    )
    stack_leaked = 'Traceback' in err_body or 'File "/' in err_body
    print(f"Error leak test: status={err_status}, stack_leaked={stack_leaked}")

    # Test 6: CORS preflight
    cors_status, cors_headers, _ = http_request(
        f"{BASE_URL}/api/health", method="OPTIONS",
        headers={"Origin": "https://evil.com", "Access-Control-Request-Method": "POST"}
    )
    cors_origin = cors_headers.get('Access-Control-Allow-Origin', '')
    cors_wildcard = cors_origin == '*'
    print(f"\nCORS preflight: origin={cors_origin}, wildcard={cors_wildcard}")

    live_pass = not xss_reflected and not traversal_success and not stack_leaked
    record("LiveServer-security", "1-8,12,16,34", live_pass,
           f"xss_reflected={xss_reflected}, traversal={traversal_success}, " +
           f"stack_leak={stack_leaked}, cors_wildcard={cors_wildcard}")
else:
    print("\nServer not running — skipping live probes.")
    print("Start with: cd /home/phuc/projects/solace-browser && python solace_browser_server.py")
    record("LiveServer-security", "1-8,12,16,34", False, "Server not running on port 9222")

In [ ]:
# =============================================================================
# Probe 23: Quantum + Crypto inventory (Floor 45: QuantumThreat + Floor 27: CryptoBreaker)
# Q: Inventory of all cryptographic primitives
# Q: Are SHA-256 and AES-256 used (quantum-resistant under current understanding)?
# =============================================================================

# Scan for all crypto primitives
crypto_patterns = {
    'SHA-256': r'sha256|SHA256|sha_256',
    'SHA-512': r'sha512|SHA512',
    'MD5': r'\bmd5\b|\bMD5\b',
    'AES-GCM': r'AESGCM|AES.GCM|aes.gcm',
    'AES-CBC': r'AES.CBC|aes.cbc',
    'RSA': r'\bRSA\b|\brsa\b',
    'ECDSA': r'\bECDSA\b|\becdsa\b',
    'PBKDF2': r'PBKDF2|pbkdf2',
    'bcrypt': r'\bbcrypt\b',
    'HMAC': r'\bhmac\b|\bHMAC\b',
}

print("Cryptographic primitive inventory:")
found_primitives = {}
for name, pattern in crypto_patterns.items():
    hits = grep_files(pattern, BROWSER_ROOT, '**/*.py')
    hits = [(f, l, t) for f, l, t in hits if '/test' not in f and '__pycache__' not in f]
    found_primitives[name] = len(hits)
    if hits:
        files = set(f.replace(BROWSER_ROOT + '/', '') for f, _, _ in hits)
        print(f"  {name}: {len(hits)} usages in {', '.join(sorted(files)[:3])}")
    else:
        print(f"  {name}: not found")

# Check for quantum-vulnerable algorithms
quantum_vulnerable = []
for algo in ['RSA', 'ECDSA']:
    if found_primitives.get(algo, 0) > 0:
        quantum_vulnerable.append(algo)

# Check for deprecated algorithms
deprecated = []
if found_primitives.get('MD5', 0) > 0:
    deprecated.append('MD5')
if found_primitives.get('AES-CBC', 0) > 0:
    deprecated.append('AES-CBC (prefer GCM)')

print(f"\nQuantum-vulnerable: {quantum_vulnerable or 'none'}")
print(f"Deprecated: {deprecated or 'none'}")
print(f"Core algorithms: SHA-256={found_primitives.get('SHA-256',0)}, AES-GCM={found_primitives.get('AES-GCM',0)}")

# PASS if using SHA-256 + AES-GCM and no RSA/ECDSA in critical paths
has_strong_crypto = found_primitives.get('SHA-256', 0) > 0 and found_primitives.get('AES-GCM', 0) > 0
record("CryptoInventory", "27,45", has_strong_crypto and len(deprecated) == 0,
       f"SHA256={found_primitives.get('SHA-256',0)}, AESGCM={found_primitives.get('AES-GCM',0)}, " +
       f"quantum_vuln={quantum_vulnerable}, deprecated={deprecated}")

In [ ]:
# =============================================================================
# Probe 24: Threat model + Security documentation (Floor 43 + 47: RedTeamLead + ThreatIntelAnalyst)
# Q: Does a formal threat model document exist?
# Q: Does a data flow diagram exist for security review?
# Q: Does the security architecture document exist as standalone artifact?
# =============================================================================

# Search for threat modeling documents
threat_docs = []
for pattern in ['**/threat*model*', '**/STRIDE*', '**/PASTA*', '**/attack*tree*',
                '**/security*architecture*', '**/data*flow*']:
    for ext in ['*.md', '*.txt', '*.json', '*.yaml']:
        combined = pattern + ext if not pattern.endswith('*') else pattern
        threat_docs.extend(Path(BROWSER_ROOT).glob(combined))

# Also check papers directory
papers_dir = Path(BROWSER_ROOT) / "papers"
security_papers = []
if papers_dir.exists():
    security_papers = list(papers_dir.glob('*security*')) + list(papers_dir.glob('*threat*'))

# Check for security diagrams
diagrams_dir = Path(BROWSER_ROOT) / "src" / "diagrams"
security_diagrams = []
if diagrams_dir.exists():
    for d in diagrams_dir.glob('*.mmd'):
        content = d.read_text(encoding='utf-8', errors='replace')[:200]
        if any(kw in content.lower() for kw in ['security', 'auth', 'oauth', 'threat', 'attack']):
            security_diagrams.append(d)

print(f"Threat model documents: {len(threat_docs)}")
for d in threat_docs:
    print(f"  {d}")

print(f"\nSecurity papers: {len(security_papers)}")
for p in security_papers:
    print(f"  {p}")

print(f"\nSecurity diagrams: {len(security_diagrams)}")
for d in security_diagrams[:5]:
    print(f"  {d.name}")

has_threat_model = len(threat_docs) > 0 or len(security_papers) > 0
record("ThreatModel-docs", "43,47", has_threat_model,
       f"threat_docs={len(threat_docs)}, papers={len(security_papers)}, diagrams={len(security_diagrams)}")

In [ ]:
# =============================================================================
# EVIDENCE SUMMARY — Final Cell
# =============================================================================

import json
from datetime import datetime, timezone

total = len(RESULTS)
passed = sum(1 for r in RESULTS if r["status"] == "PASS")
failed = sum(1 for r in RESULTS if r["status"] == "FAIL")
pass_rate = (passed / total * 100) if total > 0 else 0

print("=" * 70)
print(f"TOWER {TOWER}: {TOWER_NAME} — QA Summary")
print("=" * 70)
print(f"Total probes:  {total}")
print(f"PASSED:        {passed}")
print(f"FAILED:        {failed}")
print(f"Pass rate:     {pass_rate:.1f}%")
print(f"Target:        {MIN_SCORE}%")
print(f"Verdict:       {'MEETS TARGET' if pass_rate >= MIN_SCORE else 'BELOW TARGET'}")
print("=" * 70)

print("\nDetailed results:")
for r in RESULTS:
    icon = "[PASS]" if r["status"] == "PASS" else "[FAIL]"
    print(f"  {icon} {r['probe']} (Floor {r['floor']}): {r['detail'][:80]}")

# Critical failures
critical_fails = [r for r in RESULTS if r["status"] == "FAIL"]
if critical_fails:
    print(f"\nCritical failures ({len(critical_fails)}):")
    for r in critical_fails:
        print(f"  - {r['probe']}: {r['detail']}")

# JSON evidence bundle
evidence = {
    "tower": TOWER,
    "tower_name": TOWER_NAME,
    "northstar": NORTHSTAR,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "total_probes": total,
    "passed": passed,
    "failed": failed,
    "pass_rate": round(pass_rate, 1),
    "min_score": MIN_SCORE,
    "verdict": "MEETS TARGET" if pass_rate >= MIN_SCORE else "BELOW TARGET",
    "results": RESULTS,
}

print(f"\n--- Evidence JSON ---")
print(json.dumps(evidence, indent=2))